In [0]:
# ============================================================
# Integration (SILVER -> GOLD)
# Bike Share Toronto: Station-hour net flow enriched with:
#   - Weather (hourly)
#   - Station metadata (name, lat/lon)
#   - Public events:
#       v1: daily event signal (day-level)
#       v2: spatiotemporal station-hour event features (nearby events)
#
# Environment:
#   Databricks Serverless (Spark Connect) 
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 0) Paths (SILVER input) + GOLD output
# ============================================================
SILVER_STATION = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/station_id"
SILVER_PUBLIC  = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/public_events"
SILVER_WEATHER = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/weather_hourly"
SILVER_AGG     = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver_agg/station_hour_flow"

GOLD_DIR_V1 = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/gold/gold_v1_daily_events"
GOLD_DIR_V2 = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/gold/gold_v2_spatiotemporal_events"

# Toggle for spatiotemporal features (v2)
ENABLE_SPATIOTEMPORAL = True

# Spatiotemporal parameters
RADIUS_KM = 1.0        # event impact radius around station
EPS = 0.10             # avoids division by zero in weighted intensity
MAX_EVENT_HOURS = 48   # safety cap for bad/malformed event durations


# ============================================================
# 1) Helper functions
# ============================================================
def hour_str_to_int(col_expr):
    """
    Converts hour values like '00:00', '7:00', '07:00', '0' -> integer 0..23
    """
    return F.regexp_extract(col_expr.cast("string"), r"^(\d{1,2})", 1).cast("int")


def hour_int_to_str(col_expr):
    """
    Converts integer 0..23 -> 'HH:00'
    Used only for readability / display (not needed for joins)
    """
    return F.format_string("%02d:00", col_expr.cast("int"))


def must_have_cols(df, cols, name):
    """Raise error if expected columns are missing."""
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{name}] Missing columns: {missing}. Available: {df.columns}")


def uniqueness_report(df, keys, name, sample=20):
    """
    Data quality check:
      - Counts duplicates at the intended grain
      - Shows top duplicate groups if any exist
    """
    total = df.count()
    unique_keys = df.select(*keys).distinct().count()
    dup_df = df.groupBy(*keys).count().filter(F.col("count") > 1).orderBy(F.desc("count"))
    dup_groups = dup_df.count()
    extra_rows = (
        df.groupBy(*keys).count()
          .select(F.sum(F.col("count") - 1).alias("extra"))
          .collect()[0]["extra"]
    )
    extra_rows = int(extra_rows) if extra_rows is not None else 0

    print(f"\n=== Uniqueness report: {name} ===")
    print("Keys:", keys)
    print(f"Total rows: {total:,}")
    print(f"Distinct keys: {unique_keys:,}")
    print(f"Duplicated key groups (>1): {dup_groups:,}")
    print(f"Extra rows due to duplicates: {extra_rows:,}")
    if dup_groups > 0:
        display(dup_df.limit(sample))


def assert_no_duplicate_columns(df, name):
    """
    Spark can throw COLUMN_ALREADY_EXISTS errors if you join datasets that contain
    same column name but not in join keys. This validates BEFORE write.
    """
    cols = df.columns
    dup_cols = sorted({c for c in cols if cols.count(c) > 1})
    if dup_cols:
        raise ValueError(f"[{name}] Duplicate columns detected: {dup_cols}")


def haversine_km(lat1, lon1, lat2, lon2):
    """
    Haversine distance between two coordinates.
    Returns distance in kilometers.

    R = 6371.0 is the average Earth radius in km (standard constant).
    """
    R = 6371.0
    phi1 = F.radians(lat1)
    phi2 = F.radians(lat2)
    dphi = F.radians(lat2 - lat1)
    dlambda = F.radians(lon2 - lon1)
    a = F.pow(F.sin(dphi / 2), 2) + F.cos(phi1) * F.cos(phi2) * F.pow(F.sin(dlambda / 2), 2)
    c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
    return R * c


# ============================================================
# 2) Load SILVER (Parquet)
# ============================================================
df_flow_s    = spark.read.parquet(SILVER_AGG)
df_weather_s = spark.read.parquet(SILVER_WEATHER)
df_station_s = spark.read.parquet(SILVER_STATION)
df_events_s  = spark.read.parquet(SILVER_PUBLIC)

print("Rows df_flow   :", f"{df_flow_s.count():,}")
print("Rows df_weather:", f"{df_weather_s.count():,}")
print("Rows df_station:", f"{df_station_s.count():,}")
print("Rows df_events :", f"{df_events_s.count():,}")

# Basic schema validation (fail fast)
must_have_cols(df_flow_s,    ["station_id","year","month","day","hour","hour_str","departures","arrivals","net_flow"], "FLOW")
must_have_cols(df_weather_s, ["year","month","day","hour","temperature_2m_celsius","apparent_temperature_celsius"], "WEATHER")
must_have_cols(df_station_s, ["station_id","name","lat","lon"], "STATION")
must_have_cols(df_events_s,  ["event_id","event_name","event_category","event_start_date","event_time_starts",
                             "event_end_date","event_time_ends","latitude","longitude","mean_attendance_per_day"], "EVENTS")


# ============================================================
# 3) Standardize SILVER -> join-ready datasets
# ============================================================

# 3.1 FLOW (Backbone: station-hour)
df_flow_n = (
    df_flow_s
    .withColumn("station_id", F.col("station_id").cast("string"))
    .withColumn("year",  F.col("year").cast("int"))
    .withColumn("month", F.col("month").cast("int"))
    .withColumn("day",   F.col("day").cast("int"))
    .withColumn("hour",  F.col("hour").cast("int"))
    .withColumn("hour_str", F.col("hour_str").cast("string"))
    .withColumn("departures", F.col("departures").cast("long"))
    .withColumn("arrivals",   F.col("arrivals").cast("long"))
    .withColumn("net_flow",   F.col("net_flow").cast("long"))
    .select("station_id","year","month","day","hour","hour_str","departures","arrivals","net_flow")
)

# Ensure hour_str exists (display column only)
df_flow_n = df_flow_n.withColumn("hour_str", F.coalesce(F.col("hour_str"), hour_int_to_str(F.col("hour"))))


# 3.2 WEATHER (hour string -> hour int)
# IMPORTANT: do NOT create hour_str here to avoid duplicate columns on join.
df_weather_n = (
    df_weather_s
    .withColumn("year",  F.col("year").cast("int"))
    .withColumn("month", F.col("month").cast("int"))
    .withColumn("day",   F.col("day").cast("int"))
    .withColumn("hour",  hour_str_to_int(F.col("hour")).cast("int"))
    .withColumn("temperature_2m_celsius", F.col("temperature_2m_celsius").cast("double"))
    .withColumn("apparent_temperature_celsius", F.col("apparent_temperature_celsius").cast("double"))
    .select("year","month","day","hour","temperature_2m_celsius","apparent_temperature_celsius")
)


# 3.3 STATION
df_station_n = (
    df_station_s
    .withColumn("station_id", F.col("station_id").cast("string"))
    .withColumn("lat", F.col("lat").cast("double"))
    .withColumn("lon", F.col("lon").cast("double"))
    .select("station_id","name","lat","lon")
)


# 3.4 EVENTS: build start_ts and end_ts
ev = (
    df_events_s
    .withColumnRenamed("latitude",  "event_lat")
    .withColumnRenamed("longitude", "event_lon")
    .withColumn("event_lat", F.col("event_lat").cast("double"))
    .withColumn("event_lon", F.col("event_lon").cast("double"))
    .withColumn("mean_attendance_per_day", F.col("mean_attendance_per_day").cast("double"))
)

# Parse timestamps (date + 'HH:mm')
ev = (
    ev
    .withColumn(
        "start_ts",
        F.to_timestamp(
            F.concat_ws(" ", F.date_format("event_start_date","yyyy-MM-dd"), F.col("event_time_starts")),
            "yyyy-MM-dd HH:mm"
        )
    )
    .withColumn(
        "end_ts",
        F.to_timestamp(
            F.concat_ws(" ", F.date_format("event_end_date","yyyy-MM-dd"), F.col("event_time_ends")),
            "yyyy-MM-dd HH:mm"
        )
    )
)

# Fallbacks for missing / inconsistent timestamps
ev = ev.withColumn("end_ts", F.coalesce(F.col("end_ts"), F.expr("start_ts + interval 2 hours")))
ev = ev.withColumn("end_ts", F.when(F.col("end_ts") < F.col("start_ts"), F.col("start_ts")).otherwise(F.col("end_ts")))


# 3.4.1 EVENTS v1 (daily aggregation)
# Expands each event to all days between start_date and end_date.
# NOTE: This is day-level only (does not apply hour logic).
ev_daily = (
    ev
    .withColumn("start_date", F.to_date("start_ts"))
    .withColumn("end_date",   F.to_date("end_ts"))
    .withColumn("end_date", F.coalesce(F.col("end_date"), F.col("start_date")))
    .withColumn("date_seq", F.sequence(F.col("start_date"), F.col("end_date")))
    .withColumn("event_date", F.explode("date_seq"))
    .withColumn("year",  F.year("event_date").cast("int"))
    .withColumn("month", F.month("event_date").cast("int"))
    .withColumn("day",   F.dayofmonth("event_date").cast("int"))
    .select(
        "year","month","day",
        F.lit(1).alias("event_flag"),
        F.col("event_name").alias("event_name"),
        F.col("event_category").alias("event_category"),
        F.col("location").alias("event_location"),
        F.col("mean_attendance_per_day").cast("double").alias("event_attendance_est")
    )
)

ev_daily_agg = (
    ev_daily
    .groupBy("year","month","day")
    .agg(
        F.max("event_flag").alias("event_flag"),
        F.sum(F.coalesce(F.col("event_attendance_est"), F.lit(0.0))).alias("event_attendance_est_sum"),
        F.count(F.lit(1)).alias("events_count"),
        F.collect_set("event_category").alias("event_categories"),
        F.collect_set("event_name").alias("event_names")
    )
)


# ============================================================
# 4) Pre-join data quality (grain uniqueness)
# ============================================================
uniqueness_report(df_flow_n,    ["station_id","year","month","day","hour"], "FLOW")
uniqueness_report(df_weather_n, ["year","month","day","hour"],             "WEATHER")
uniqueness_report(df_station_n, ["station_id"],                           "STATION")
uniqueness_report(ev_daily_agg, ["year","month","day"],                   "EVENTS_DAILY_AGG")


# ============================================================
# 5) GOLD_V1 integration (day-level event baseline)
# ============================================================
df_gold_v1 = (
    df_flow_n
    .join(df_weather_n, on=["year","month","day","hour"], how="left")
    .join(df_station_n, on="station_id", how="left")
    .join(ev_daily_agg, on=["year","month","day"], how="left")
)

# Fill event defaults (no event day => 0)
df_gold_v1 = df_gold_v1.na.fill({
    "event_flag": 0,
    "event_attendance_est_sum": 0.0,
    "events_count": 0
})

# Ensure arrays are never null
df_gold_v1 = df_gold_v1.withColumn("event_categories", F.coalesce(F.col("event_categories"), F.array()))
df_gold_v1 = df_gold_v1.withColumn("event_names",      F.coalesce(F.col("event_names"),      F.array()))

assert_no_duplicate_columns(df_gold_v1, "GOLD_V1_PREWRITE")

print("Gold v1 preview:")
display(df_gold_v1.limit(10))


# ============================================================
# 6) GOLD_V2: Spatiotemporal station-hour event features
# ============================================================
if ENABLE_SPATIOTEMPORAL:

    # 6.1 Expand events into hours between start_ts and end_ts.
    # SAFETY:
    #   MAX_EVENT_HOURS caps the duration to avoid malformed records
    #   (e.g., events with end before start, or huge durations).
    ev_h = (
        ev
        .withColumn("start_h", F.date_trunc("hour", F.col("start_ts")))
        .withColumn("end_h",   F.date_trunc("hour", F.col("end_ts")))
        .withColumn("hours_len", (F.unix_timestamp("end_h") - F.unix_timestamp("start_h")) / 3600)
        .filter(F.col("start_h").isNotNull() & F.col("end_h").isNotNull())
        .filter(F.col("hours_len") >= 0)
        .filter(F.col("hours_len") <= F.lit(MAX_EVENT_HOURS))
        .withColumn("hour_seq", F.sequence(F.col("start_h"), F.col("end_h"), F.expr("interval 1 hour")))
        .withColumn("event_hour_ts", F.explode("hour_seq"))
        .withColumn("year",  F.year("event_hour_ts").cast("int"))
        .withColumn("month", F.month("event_hour_ts").cast("int"))
        .withColumn("day",   F.dayofmonth("event_hour_ts").cast("int"))
        .withColumn("hour",  F.hour("event_hour_ts").cast("int"))
        .select(
            "event_id","event_name","event_category","location",
            "event_lat","event_lon",
            "mean_attendance_per_day",
            "year","month","day","hour"
        )
    )

    # 6.2 Station coordinates subset (only stations with valid geo)
    st = (
        df_station_n
        .select("station_id",
                F.col("lat").alias("station_lat"),
                F.col("lon").alias("station_lon"))
        .filter(F.col("station_lat").isNotNull() & F.col("station_lon").isNotNull())
    )

    # 6.3 Spatial join approach:
    # We cross-join station x hourly-events and filter by distance <= RADIUS_KM.
    # We broadcast events because events/hour dataset is typically much smaller than stations.
    ev_station_hour = (
        st.crossJoin(F.broadcast(ev_h))
          .withColumn("distance_km", haversine_km(F.col("station_lat"), F.col("station_lon"),
                                                 F.col("event_lat"),   F.col("event_lon")))
          .filter(F.col("distance_km") <= F.lit(RADIUS_KM))
    )

    # 6.4 Aggregate to station-hour grain
    # event_weighted_intensity sums attendance/distance (inverse distance weighting)
    event_features = (
        ev_station_hour
        .withColumn("event_active_nearby_flag", F.lit(1))
        .withColumn("attendance_est", F.col("mean_attendance_per_day").cast("double"))
        .withColumn("weighted_intensity", F.col("attendance_est") / (F.col("distance_km") + F.lit(EPS)))
        .groupBy("station_id","year","month","day","hour")
        .agg(
            F.max("event_active_nearby_flag").alias("event_active_nearby_flag"),
            F.countDistinct("event_id").alias("events_nearby_count"),
            F.min("distance_km").alias("nearest_event_km"),
            F.sum("weighted_intensity").alias("event_weighted_intensity"),
            F.sum("attendance_est").alias("event_attendance_est_sum_nearby"),
            F.collect_set("event_category").alias("event_categories_nearby"),
            F.collect_set("event_name").alias("event_names_nearby")
        )
    )

    uniqueness_report(event_features, ["station_id","year","month","day","hour"], "EVENT_FEATURES_STATION_HOUR")

    # 6.5 Final join: backbone FLOW + WEATHER + STATION + DAILY events + STATION-HOUR event_features
    df_gold_v2 = (
        df_flow_n
        .join(df_weather_n, on=["year","month","day","hour"], how="left")
        .join(df_station_n, on="station_id", how="left")
        .join(ev_daily_agg, on=["year","month","day"], how="left")  # day baseline signal
        .join(event_features, on=["station_id","year","month","day","hour"], how="left")  # spatiotemporal
    )

    # Default values for NULLs (no event nearby)
    df_gold_v2 = df_gold_v2.na.fill({
        "event_flag": 0,
        "event_attendance_est_sum": 0.0,
        "events_count": 0,
        "event_active_nearby_flag": 0,
        "events_nearby_count": 0,
        "nearest_event_km": 999.0,  # sentinel for "no nearby event"
        "event_weighted_intensity": 0.0,
        "event_attendance_est_sum_nearby": 0.0
    })

    # Ensure arrays are never null
    df_gold_v2 = df_gold_v2.withColumn("event_categories",        F.coalesce(F.col("event_categories"),        F.array()))
    df_gold_v2 = df_gold_v2.withColumn("event_names",             F.coalesce(F.col("event_names"),             F.array()))
    df_gold_v2 = df_gold_v2.withColumn("event_categories_nearby", F.coalesce(F.col("event_categories_nearby"), F.array()))
    df_gold_v2 = df_gold_v2.withColumn("event_names_nearby",      F.coalesce(F.col("event_names_nearby"),      F.array()))

    assert_no_duplicate_columns(df_gold_v2, "GOLD_V2_PREWRITE")

    print("Gold v2 preview:")
    display(df_gold_v2.limit(10))


# ============================================================
# 7) Post-join DQ checks (must pass)
# ============================================================
def dq_checks(df, name):
    print(f"\n==================== DQ CHECKS: {name} ====================")

    grain = ["station_id","year","month","day","hour"]

    # 7.1 Grain uniqueness
    dups = df.groupBy(*grain).count().filter(F.col("count") > 1)
    dup_count = dups.count()
    print("Duplicate grain rows:", dup_count)
    if dup_count > 0:
        display(dups.orderBy(F.desc("count")).limit(20))
        raise ValueError(f"{name}: Duplicate grain detected!")

    # 7.2 Null key fields
    nulls = df.select(
        F.count(F.when(F.col("station_id").isNull(), 1)).alias("null_station_id"),
        F.count(F.when(F.col("year").isNull(), 1)).alias("null_year"),
        F.count(F.when(F.col("month").isNull(), 1)).alias("null_month"),
        F.count(F.when(F.col("day").isNull(), 1)).alias("null_day"),
        F.count(F.when(F.col("hour").isNull(), 1)).alias("null_hour"),
    )
    display(nulls)

    # 7.3 Coverage checks
    weather_cov = df.select(
        F.count("*").alias("rows_total"),
        F.count(F.when(F.col("temperature_2m_celsius").isNotNull(), 1)).alias("rows_with_weather")
    )
    station_cov = df.select(
        F.count("*").alias("rows_total"),
        F.count(F.when(F.col("lat").isNotNull() & F.col("lon").isNotNull(), 1)).alias("rows_with_station_geo")
    )
    display(weather_cov)
    display(station_cov)

    # 7.4 Event baseline coverage
    if "event_flag" in df.columns:
        display(df.groupBy("event_flag").count().orderBy("event_flag"))

    # 7.5 Sanity ranges
    bad = df.filter(
        (~F.col("hour").between(0,23)) |
        (~F.col("month").between(1,12)) |
        (~F.col("day").between(1,31))
    )
    bad_count = bad.count()
    print("Bad date/hour rows:", bad_count)
    if bad_count > 0:
        display(bad.select(*grain).limit(50))
        raise ValueError(f"{name}: Sanity range violations!")


dq_checks(df_gold_v1, "GOLD_V1")
if ENABLE_SPATIOTEMPORAL:
    dq_checks(df_gold_v2, "GOLD_V2")


# ============================================================
# 8) Write GOLD datasets (partitionBy year/month)
# ============================================================
def write_gold(df, out_dir, name):
    """
    Writes parquet partitioned by (year, month).
    Overwrite mode is used because GOLD is a rebuilt dataset.
    """
    df_write = df.filter(
        F.col("station_id").isNotNull() &
        F.col("year").isNotNull() &
        F.col("month").isNotNull() &
        F.col("day").isNotNull() &
        F.col("hour").isNotNull()
    )

    assert_no_duplicate_columns(df_write, f"{name}_WRITE")

    (df_write
     .write
     .mode("overwrite")
     .partitionBy("year","month")
     .parquet(out_dir)
    )
    print(f"Written {name} to: {out_dir}")


write_gold(df_gold_v1, GOLD_DIR_V1, "GOLD_V1")
if ENABLE_SPATIOTEMPORAL:
    write_gold(df_gold_v2, GOLD_DIR_V2, "GOLD_V2")


# ============================================================
# 9) Post-write validation (read back + sample + grain check)
# ============================================================
def post_write_validation(out_dir, name):
    df_read = spark.read.parquet(out_dir)
    print(f"\n{name} read rows:", f"{df_read.count():,}")
    df_read.printSchema()

    display(df_read.groupBy("year","month").count().orderBy("year","month"))
    display(df_read.orderBy("year","month","day","hour","station_id").limit(10))

    w = Window.partitionBy("year","month").orderBy("day","hour","station_id")
    sample_10 = (
        df_read
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") <= 10)
        .drop("rn")
        .orderBy("year","month","day","hour","station_id")
    )
    display(sample_10)

    uniqueness_report(df_read, ["station_id","year","month","day","hour"], f"{name}_POSTREAD_GRAIN")


post_write_validation(GOLD_DIR_V1, "GOLD_V1")
if ENABLE_SPATIOTEMPORAL:
    post_write_validation(GOLD_DIR_V2, "GOLD_V2")


# ============================================================
# 10) Post-processing on GOLD_V2:
#     - Rename daily event columns (avoid confusion)
#     - Create operational impact categories & numeric score
#     - Overwrite GOLD_V2 with updated columns
# ============================================================
if ENABLE_SPATIOTEMPORAL:

    # 10.1 Rename daily event columns to make intent explicit
    rename_map = {
        "event_flag": "event_day_flag",
        "event_attendance_est_sum": "event_day_attendance_sum",
        "events_count": "events_day_count",
        "event_categories": "event_day_categories",
        "event_names": "event_day_names",
    }

    for old, new in rename_map.items():
        if old in df_gold_v2.columns and new not in df_gold_v2.columns:
            df_gold_v2 = df_gold_v2.withColumnRenamed(old, new)

    # 10.2 Operational thresholds for event impact (station-hour, nearby)
    # Based on event_attendance_est_sum_nearby (already spatial + hourly).
    att  = F.coalesce(F.col("event_attendance_est_sum_nearby"), F.lit(0.0)).cast("double")
    flag = F.coalesce(F.col("event_active_nearby_flag"), F.lit(0)).cast("int")

    df_gold_v2 = (
        df_gold_v2
        .withColumn("event_active_nearby_flag", flag)
        .withColumn("event_attendance_est_sum_nearby", att)

        .withColumn(
            "event_impact_category",
            F.when(flag == 0, F.lit("No Event"))
             .when(att <= 5000,   F.lit("Low"))
             .when(att <= 25000,  F.lit("Small"))
             .when(att <= 100000, F.lit("Medium"))
             .when(att <= 500000, F.lit("High"))
             .otherwise(F.lit("Very High"))
        )

        .withColumn(
            "event_impact_score",
            F.when(flag == 0, F.lit(0))
             .when(att <= 5000,   F.lit(1))
             .when(att <= 25000,  F.lit(2))
             .when(att <= 100000, F.lit(3))
             .when(att <= 500000, F.lit(4))
             .otherwise(F.lit(5))
             .cast("int")
        )
    )

    # 10.3 Quick validations
    (df_gold_v2
     .groupBy("event_active_nearby_flag", "event_impact_category")
     .count()
     .orderBy("event_active_nearby_flag", "event_impact_category")
     .display()
    )

    (df_gold_v2
     .groupBy("event_impact_score")
     .count()
     .orderBy("event_impact_score")
     .display()
    )

    (df_gold_v2
     .groupBy("event_active_nearby_flag","event_impact_category","event_impact_score")
     .count()
     .orderBy("event_active_nearby_flag","event_impact_score")
     .display()
    )

    # 10.4 Persist updated GOLD_V2
    # NOTE:
    # This overwrites the existing GOLD_V2 folder with the new columns.
    (df_gold_v2
     .write
     .mode("overwrite")
     .partitionBy("year","month")
     .parquet(GOLD_DIR_V2)
    )

    print("Updated GOLD_V2 written with event impact category & score.")
    spark.read.parquet(GOLD_DIR_V2).printSchema()


Rows df_flow   : 5,725,483
Rows df_weather: 17,544
Rows df_station: 1,008
Rows df_events : 240

=== Uniqueness report: FLOW ===
Keys: ['station_id', 'year', 'month', 'day', 'hour']
Total rows: 5,725,483
Distinct keys: 5,725,483
Duplicated key groups (>1): 0
Extra rows due to duplicates: 0

=== Uniqueness report: WEATHER ===
Keys: ['year', 'month', 'day', 'hour']
Total rows: 17,544
Distinct keys: 17,544
Duplicated key groups (>1): 0
Extra rows due to duplicates: 0

=== Uniqueness report: STATION ===
Keys: ['station_id']
Total rows: 1,008
Distinct keys: 1,008
Duplicated key groups (>1): 0
Extra rows due to duplicates: 0

=== Uniqueness report: EVENTS_DAILY_AGG ===
Keys: ['year', 'month', 'day']
Total rows: 193
Distinct keys: 193
Duplicated key groups (>1): 0
Extra rows due to duplicates: 0
Gold v1 preview:


year,month,day,station_id,hour,hour_str,departures,arrivals,net_flow,temperature_2m_celsius,apparent_temperature_celsius,name,lat,lon,event_flag,event_attendance_est_sum,events_count,event_categories,event_names
2024,7,13,7202,15,15:00,15,10,-5,25.4,29.2,York St / Queen St W (City Hall),43.65160619179464,-79.38404693334348,0,0.0,0,List(),List()
2024,7,18,7534,13,13:00,4,4,0,19.0,17.9,Walnut Ave / Queen St W,43.645469,-79.411084,1,37500.0,1,List(Arts),List(Tirgan Festival)
2024,7,22,7323,8,08:00,3,5,2,16.0,15.4,457 King St W,43.6452091,-79.3960744,0,0.0,0,List(),List()
2024,7,17,7320,15,15:00,6,4,-2,25.3,26.8,Front St W / University Ave (1),43.6451635,-79.3831757,0,0.0,0,List(),List()
2024,7,5,7257,8,08:00,3,7,4,19.0,20.2,Dundas St W / St. Patrick St,43.6545174,-79.3895315,0,0.0,0,List(),List()
2024,7,11,7169,15,15:00,4,4,0,22.8,22.9,Front St W / Bay St (North Side),43.646162,-79.378912,0,0.0,0,List(),List()
2024,7,1,7822,22,22:00,49,45,-4,22.8,22.7,Leslie St / Commissioners St,43.657275497390124,-79.32661544194183,1,120000.0,1,List(Holiday),List(Canada Day 2024)
2024,7,2,7668,13,13:00,5,3,-2,19.6,20.4,Simcoe St / Dundas St W,43.655103,-79.389295,0,0.0,0,List(),List()
2024,7,8,7180,19,19:00,2,1,-1,26.8,28.3,Lansdowne Subway Station,43.65922918158098,-79.44320901022282,0,0.0,0,List(),List()
2024,7,17,7296,8,08:00,3,1,-2,20.3,21.5,Westmoreland Ave / Fernbank Ave,43.665327,-79.43235,0,0.0,0,List(),List()



=== Uniqueness report: EVENT_FEATURES_STATION_HOUR ===
Keys: ['station_id', 'year', 'month', 'day', 'hour']
Total rows: 83,317
Distinct keys: 83,317
Duplicated key groups (>1): 0
Extra rows due to duplicates: 0
Gold v2 preview:


station_id,year,month,day,hour,hour_str,departures,arrivals,net_flow,temperature_2m_celsius,apparent_temperature_celsius,name,lat,lon,event_flag,event_attendance_est_sum,events_count,event_categories,event_names,event_active_nearby_flag,events_nearby_count,nearest_event_km,event_weighted_intensity,event_attendance_est_sum_nearby,event_categories_nearby,event_names_nearby
7180,2024,7,8,19,19:00,2,1,-1,26.8,28.3,Lansdowne Subway Station,43.65922918158098,-79.44320901022282,0,0.0,0,List(),List(),0,0,999.0,0.0,0.0,List(),List()
7320,2024,7,17,15,15:00,6,4,-2,25.3,26.8,Front St W / University Ave (1),43.6451635,-79.3831757,0,0.0,0,List(),List(),0,0,999.0,0.0,0.0,List(),List()
7668,2024,7,2,13,13:00,5,3,-2,19.6,20.4,Simcoe St / Dundas St W,43.655103,-79.389295,0,0.0,0,List(),List(),0,0,999.0,0.0,0.0,List(),List()
7202,2024,7,13,15,15:00,15,10,-5,25.4,29.2,York St / Queen St W (City Hall),43.65160619179464,-79.38404693334348,0,0.0,0,List(),List(),0,0,999.0,0.0,0.0,List(),List()
7534,2024,7,18,13,13:00,4,4,0,19.0,17.9,Walnut Ave / Queen St W,43.645469,-79.411084,1,37500.0,1,List(Arts),List(Tirgan Festival),0,0,999.0,0.0,0.0,List(),List()
7822,2024,7,1,22,22:00,49,45,-4,22.8,22.7,Leslie St / Commissioners St,43.657275497390124,-79.32661544194183,1,120000.0,1,List(Holiday),List(Canada Day 2024),0,0,999.0,0.0,0.0,List(),List()
7169,2024,7,11,15,15:00,4,4,0,22.8,22.9,Front St W / Bay St (North Side),43.646162,-79.378912,0,0.0,0,List(),List(),0,0,999.0,0.0,0.0,List(),List()
7296,2024,7,17,8,08:00,3,1,-2,20.3,21.5,Westmoreland Ave / Fernbank Ave,43.665327,-79.43235,0,0.0,0,List(),List(),0,0,999.0,0.0,0.0,List(),List()
7323,2024,7,22,8,08:00,3,5,2,16.0,15.4,457 King St W,43.6452091,-79.3960744,0,0.0,0,List(),List(),0,0,999.0,0.0,0.0,List(),List()
7257,2024,7,5,8,08:00,3,7,4,19.0,20.2,Dundas St W / St. Patrick St,43.6545174,-79.3895315,0,0.0,0,List(),List(),0,0,999.0,0.0,0.0,List(),List()



==================== DQ CHECKS: GOLD_V1 ====================
Duplicate grain rows: 0


...


=== Uniqueness report: GOLD_V2_POSTREAD_GRAIN ===
Keys: ['station_id', 'year', 'month', 'day', 'hour']
Total rows: 5,725,483
Distinct keys: 5,725,483
Duplicated key groups (>1): 0
Extra rows due to duplicates: 0


event_active_nearby_flag,event_impact_category,count
0,No Event,5670511
1,High,5447
1,Low,2172
1,Medium,22718
1,Small,19056
1,Very High,5579


event_impact_score,count
0,5670511
1,2172
2,19056
3,22718
4,5447
5,5579


event_active_nearby_flag,event_impact_category,event_impact_score,count
0,No Event,0,5670511
1,Low,1,2172
1,Small,2,19056
1,Medium,3,22718
1,High,4,5447
1,Very High,5,5579


Updated GOLD_V2 written with event impact category & score.
root
 |-- station_id: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- hour_str: string (nullable = true)
 |-- departures: long (nullable = true)
 |-- arrivals: long (nullable = true)
 |-- net_flow: long (nullable = true)
 |-- temperature_2m_celsius: double (nullable = true)
 |-- apparent_temperature_celsius: double (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- event_day_flag: integer (nullable = true)
 |-- event_day_attendance_sum: double (nullable = true)
 |-- events_day_count: long (nullable = true)
 |-- event_day_categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- event_day_names: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- event_active_nearby_flag: integer (nullable = true)
 |-- events_nearby_count: long (nullable = tr